## Things the docs don't shout about

Spark works the way you expect — until it doesn't. Most surprises aren't bugs; they're consequences of laziness, JVM semantics, distributed execution, or SQL null handling leaking into the DataFrame API.

This notebook collects the traps that beginners and even experienced users hit repeatedly. Each section is a tiny demo of something that **looks correct, runs without error, and silently gives the wrong answer or wastes minutes of compute**.

```text
Categories:
  · Lazy evaluation  → cache, re-computation, withColumn-in-a-loop
  · Null semantics   → comparisons, joins, filters
  · Schema gotchas   → union vs unionByName, decimal overflow, date parsing
  · API confusables  → repartition vs coalesce vs partitionBy
  · Builders/session → getOrCreate stickiness, classic vs Connect builder
  · Driver pitfalls  → collect, toPandas, Python UDFs
```


## Setup

Small synthetic DataFrames so every section is self-contained. No `data/` folder required.


In [1]:
import os, time
from decimal import Decimal
from pyspark.sql import SparkSession, Row, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DecimalType, DateType, TimestampType,
)

spark = (SparkSession.builder
    .appName("ImportantNotes")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Tiny customer / transaction tables used by several demos
customers = spark.createDataFrame([
    ("C1", "Alice",   "NY",   720),
    ("C2", "Bob",     "CA",   None),     # null credit_score
    ("C3", "Carol",   None,   810),      # null state
    ("C4", "Dan",     "TX",   650),
    ("C5", "Eve",     "NY",   None),
], "customer_id STRING, name STRING, state STRING, credit_score INT")

txns = spark.createDataFrame([
    ("T1", "C1", 120.00),
    ("T2", "C1",  45.50),
    ("T3", "C2", 999.00),
    ("T4", "C4",  10.00),
    ("T5", None,  77.00),      # orphan txn (null customer_id)
], "txn_id STRING, customer_id STRING, amount DOUBLE")

print(f"Spark {spark.version} ready  ·  customers={customers.count()}  txns={txns.count()}")


26/05/12 09:14:34 WARN Utils: Your hostname, maddipotis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.181 instead (on interface en0)
26/05/12 09:14:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/12 09:14:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 ready  ·  customers=5  txns=5


## 1 — Every action re-runs the DAG

DataFrame transformations are lazy. **Actions** (`count`, `collect`, `write`, `show`, `toPandas`) trigger execution — and each one executes the **entire lineage from the source**.

That's the trap: if you call `count()` for a log message and `write()` to persist, Spark reads the source file twice and recomputes every transformation twice.

The fix is `.cache()` (or `.persist()`) so subsequent actions reuse the materialized result.


In [2]:
# Build an expensive-ish DataFrame
heavy = (txns
    .filter(F.col("amount") > 0)
    .withColumn("amount_x2", F.col("amount") * 2)
    .withColumn("amount_log", F.log("amount")))

# Two actions — without cache, the lineage runs twice
t0 = time.time(); n  = heavy.count()
t1 = time.time(); s  = heavy.agg(F.sum("amount_x2")).first()[0]
print(f"Without cache : count={n}, sum={s:.2f}  ({t1-t0:.2f}s + {time.time()-t1:.2f}s)")

# Cache and repeat — second action reads from the cache
heavy.cache()
heavy.count()                                 # materializes the cache
t0 = time.time(); s = heavy.agg(F.sum("amount_x2")).first()[0]
print(f"With cache    : sum={s:.2f}           ({time.time()-t0:.2f}s)")
heavy.unpersist()


Without cache : count=5, sum=2503.00  (0.08s + 0.10s)


With cache    : sum=2503.00           (0.07s)


DataFrame[txn_id: string, customer_id: string, amount: double, amount_x2: double, amount_log: double]

## 2 — `cache()` is also lazy

Calling `.cache()` does **not** materialize the DataFrame. It only marks the DataFrame so that the *next* action will materialize and store it. Until then, the data is not cached.

A common mistake: cache a DataFrame, immediately inspect Spark UI, see nothing in the Storage tab, conclude that caching is broken.


In [3]:
df = txns.filter(F.col("amount") > 50)
df.cache()                                    # marks but does not materialize

print(f"isCached flag set : {df.is_cached}")
print(f"Storage level     : {df.storageLevel}")

# Trigger an action — NOW the cache is populated
_ = df.count()
print("After first action — cache is populated and visible in Spark UI's Storage tab")
df.unpersist()


isCached flag set : True
Storage level     : Disk Memory Deserialized 1x Replicated
After first action — cache is populated and visible in Spark UI's Storage tab


DataFrame[txn_id: string, customer_id: string, amount: double]

## 3 — `col == null` returns null, not false

SQL three-valued logic leaks into the DataFrame API. Any comparison involving `null` returns `null`, and `null` is **not** truthy, so `filter` and `where` silently drop those rows.

| Predicate | Behaviour on null |
|---|---|
| `col == None` | Always returns null — matches nothing |
| `col != value` | Drops null rows (null != value → null) |
| `col.isNull()` | Correct way to find nulls |
| `col.eqNullSafe(other)` (`<=>`) | Treats null = null as true |


In [4]:
# Wrong: trying to find rows where credit_score IS null
wrong = customers.filter(F.col("credit_score") == None)
print(f"customers.filter(col == None) → {wrong.count()} rows  (always 0)")

# Right: use isNull
right = customers.filter(F.col("credit_score").isNull())
print(f"customers.filter(col.isNull()) → {right.count()} rows")

# Equally surprising: filter on != drops nulls
not_720 = customers.filter(F.col("credit_score") != 720)
print(f"credit_score != 720 → {not_720.count()} rows  "
      f"(null rows silently dropped; expected {customers.count() - 1}, got {not_720.count()})")

# Null-safe equality keeps nulls
print("eqNullSafe demo:")
customers.withColumn("is_unknown",
    F.col("credit_score").eqNullSafe(F.lit(None))
).select("name", "credit_score", "is_unknown").show()


customers.filter(col == None) → 0 rows  (always 0)
customers.filter(col.isNull()) → 2 rows


credit_score != 720 → 2 rows  (null rows silently dropped; expected 4, got 2)
eqNullSafe demo:


+-----+------------+----------+
| name|credit_score|is_unknown|
+-----+------------+----------+
|Alice|         720|     false|
|  Bob|        NULL|      true|
|Carol|         810|     false|
|  Dan|         650|     false|
|  Eve|        NULL|      true|
+-----+------------+----------+



## 4 — Joining on a nullable column drops null-key rows

The same null semantics that hide `null != 720` also affect joins. `null = null` is `null`, not `true`, so rows with null join keys disappear from every join type except a deliberate null-safe one.

For inner joins this is usually what you want. For outer joins where you expect orphan rows to survive, use `eqNullSafe` or filter nulls explicitly before joining.


In [5]:
# Txn T5 has a null customer_id — an "orphan"
joined = txns.join(customers, on="customer_id", how="left")
print(f"LEFT JOIN on customer_id → {joined.count()} rows")
print(f"Txns input: {txns.count()} — the null-key row T5 is GONE from a left join too:")
joined.select("txn_id", "customer_id", "name", "amount").show()

# Why? Because the join key in txns is null and null = null returns null
print("Workaround: filter nulls before joining, or coalesce to a sentinel:")
fixed = (txns
    .withColumn("customer_id", F.coalesce(F.col("customer_id"), F.lit("UNKNOWN")))
    .join(customers, on="customer_id", how="left"))
fixed.select("txn_id", "customer_id", "name").show()


LEFT JOIN on customer_id → 5 rows


Txns input: 5 — the null-key row T5 is GONE from a left join too:
+------+-----------+-----+------+
|txn_id|customer_id| name|amount|
+------+-----------+-----+------+
|    T1|         C1|Alice| 120.0|
|    T2|         C1|Alice|  45.5|
|    T3|         C2|  Bob| 999.0|
|    T4|         C4|  Dan|  10.0|
|    T5|       NULL| NULL|  77.0|
+------+-----------+-----+------+

Workaround: filter nulls before joining, or coalesce to a sentinel:


+------+-----------+-----+
|txn_id|customer_id| name|
+------+-----------+-----+
|    T1|         C1|Alice|
|    T2|         C1|Alice|
|    T3|         C2|  Bob|
|    T4|         C4|  Dan|
|    T5|    UNKNOWN| NULL|
+------+-----------+-----+



## 5 — `union` is positional; `unionByName` matches by column name

`df.union(other)` ignores column names and stacks rows by **position**. If two DataFrames have the same columns in a different order, `union` silently produces garbage — values land under the wrong header.

Always prefer `unionByName`, which matches by column name and (optionally) fills missing columns with null via `allowMissingColumns=True`.


In [6]:
a = spark.createDataFrame([("CUST1", "Alice", 720)],
                          "customer_id STRING, name STRING, score INT")
b = spark.createDataFrame([(810, "Bob", "CUST2")],
                          "score INT, name STRING, customer_id STRING")

print("union (positional) — CORRUPTED rows:")
a.union(b).show()

print("unionByName (correct):")
a.unionByName(b).show()

# Missing columns? unionByName can fill with null
c = spark.createDataFrame([("CUST3", "Carol")], "customer_id STRING, name STRING")
print("unionByName with allowMissingColumns:")
a.unionByName(c, allowMissingColumns=True).show()


union (positional) — CORRUPTED rows:
+-----------+-----+-----+
|customer_id| name|score|
+-----------+-----+-----+
|      CUST1|Alice|  720|
|        810|  Bob|CUST2|
+-----------+-----+-----+

unionByName (correct):
+-----------+-----+-----+
|customer_id| name|score|
+-----------+-----+-----+
|      CUST1|Alice|  720|
|      CUST2|  Bob|  810|
+-----------+-----+-----+



unionByName with allowMissingColumns:


+-----------+-----+-----+
|customer_id| name|score|
+-----------+-----+-----+
|      CUST1|Alice|  720|
|      CUST3|Carol| NULL|
+-----------+-----+-----+



## 6 — `monotonically_increasing_id()` is monotonic, not sequential

A frequent misuse: someone needs an auto-incrementing primary key, reaches for `monotonically_increasing_id()`, and gets values like `8589934592`, `8589934593`, `17179869184` — large gaps, not 1, 2, 3.

The function only guarantees:
- Unique within a job
- Monotonically increasing within each partition

The high bits encode the partition number, so values from partition N start way above values from partition N-1. For a true row sequence, use `row_number().over(Window.orderBy(...))` — but be aware that a global ordering forces all rows through one partition.


In [7]:
# 3 partitions, 5 rows
df = spark.range(5).repartition(3).withColumn("monotonic_id", F.monotonically_increasing_id())
print("monotonically_increasing_id — gaps reveal partition boundaries:")
df.orderBy("id").show()

# Contiguous IDs need a Window with global order
w = Window.orderBy("id")
print("row_number — truly contiguous (but forces single-partition sort):")
df.select("id", F.row_number().over(w).alias("rn")).orderBy("id").show()


monotonically_increasing_id — gaps reveal partition boundaries:


+---+------------+
| id|monotonic_id|
+---+------------+
|  0| 17179869184|
|  1| 17179869185|
|  2| 17179869186|
|  3|           0|
|  4|           1|
+---+------------+

row_number — truly contiguous (but forces single-partition sort):


+---+---+
| id| rn|
+---+---+
|  0|  1|
|  1|  2|
|  2|  3|
|  3|  4|
|  4|  5|
+---+---+



## 7 — `when().otherwise()` evaluates every branch

In Python, `1 / 0 if False else 0` is safe — the false branch is never evaluated. In Spark, `when(cond, expr1).otherwise(expr2)` does **not** short-circuit. Both `expr1` and `expr2` are computed for every row, then the result is selected.

That means a guarded division still touches the divisor. Spark itself returns null for divide-by-zero (instead of raising), so you don't always see the failure — but the cost is paid, and if you replace division with something that *does* throw (e.g. a UDF), every row hits the failure branch even when the guard would skip it.


In [8]:
df = spark.createDataFrame([(10, 2), (20, 0), (30, 5)], "num INT, den INT")

# Guard with when — but both branches still evaluate
guarded = df.withColumn("ratio",
    F.when(F.col("den") != 0, F.col("num") / F.col("den")).otherwise(F.lit(None)))
guarded.show()
print("Spark returns null for divide-by-zero rather than raising, masking the issue.")
print("In a Python UDF, BOTH branches would execute → exceptions even when guarded.\n")

# Safer pattern: turn the zero divisor into null so the division itself yields null
denom_or_null = F.when(F.col("den") == 0, F.lit(None)).otherwise(F.col("den"))
safer = df.withColumn("ratio", F.col("num") / denom_or_null)
safer.show()


+---+---+-----+
|num|den|ratio|
+---+---+-----+
| 10|  2|  5.0|
| 20|  0| NULL|
| 30|  5|  6.0|
+---+---+-----+

Spark returns null for divide-by-zero rather than raising, masking the issue.
In a Python UDF, BOTH branches would execute → exceptions even when guarded.



+---+---+-----+
|num|den|ratio|
+---+---+-----+
| 10|  2|  5.0|
| 20|  0| NULL|
| 30|  5|  6.0|
+---+---+-----+



## 8 — `collect()` and `toPandas()` pull everything to the driver

Every distributed Spark API has a cliff: the moment you call `collect()`, `toPandas()`, or `toLocalIterator()` (kind of), all selected rows leave the executors and land in the driver's heap. On a 200 GB DataFrame with a 4 GB driver, this OOMs the driver and kills the job.

| Action | Where the data ends up |
|---|---|
| `count`, `sum`, `agg` | Driver gets a scalar — safe |
| `show(n)`, `take(n)`, `head(n)` | Driver gets `n` rows — safe |
| `collect`, `toPandas` | Driver gets **all** rows — danger |
| `toLocalIterator` | Driver streams partitions one-at-a-time — safer than collect for medium-large data |

If you only need a peek, use `show` or `take`. If you genuinely need pandas for a small aggregate result, aggregate **on Spark first**, then `.toPandas()`.


In [9]:
# Safe: action that returns a scalar
print(f"Total amount (scalar): {txns.agg(F.sum('amount')).first()[0]}")

# Safe: bounded peek
print(f"head(2): {txns.head(2)}")

# Risky pattern (looks innocent on small data, kills the driver on big data)
df_small = txns                   # imagine this is 500 GB
all_rows = df_small.collect()     # <- entire dataset to driver
print(f"collect() returned {len(all_rows)} rows  (fine here; would OOM at scale)")

# Better: aggregate first, then collect / toPandas
agg = (txns.groupBy("customer_id").agg(F.sum("amount").alias("total")))
print("Aggregate-then-toPandas (driver sees only group rows):")
print(agg.toPandas())


Total amount (scalar): 1251.5


head(2): [Row(txn_id='T1', customer_id='C1', amount=120.0), Row(txn_id='T2', customer_id='C1', amount=45.5)]
collect() returned 5 rows  (fine here; would OOM at scale)
Aggregate-then-toPandas (driver sees only group rows):


  customer_id  total
0          C1  165.5
1          C2  999.0
2          C4   10.0
3        None   77.0


## 9 — Python UDFs are a Catalyst black box

A Python UDF runs in a Python worker process: Spark serializes each row from the JVM to Python, calls your function, serializes the result back. Catalyst can't push the UDF into the data source, can't reorder predicates around it, and pays a per-row serialization cost.

| Choice | Speed | Catalyst sees |
|---|---|---|
| Built-in (`F.upper`, `F.when`, arithmetic) | Fastest | Full expression — optimized, vectorized |
| Pandas UDF / Arrow UDF | Fast | Opaque, but processes batches at a time |
| Python UDF (`udf(...)`) | 10–100× slower | Opaque, **row-by-row** |
| Scala/Java UDF | Native speed | Opaque |

Reach for a Python UDF only when you genuinely cannot express the logic in built-ins.


In [10]:
big = spark.range(2_000_000)

# Built-in: stays in the JVM, Catalyst optimizes it
t0 = time.time()
n1 = big.withColumn("y", F.col("id") * 2 + 1).filter(F.col("y") > 1000).count()
t_builtin = time.time() - t0

# Python UDF: every row crosses the JVM ↔ Python boundary
@F.udf("long")
def slow_double_plus_one(x):
    return x * 2 + 1

t0 = time.time()
n2 = big.withColumn("y", slow_double_plus_one("id")).filter(F.col("y") > 1000).count()
t_udf = time.time() - t0

print(f"Built-in : {t_builtin:.2f}s  ({n1} rows)")
print(f"Python UDF: {t_udf:.2f}s  ({n2} rows)  → {t_udf/t_builtin:.1f}× slower")


Built-in : 0.10s  (1999500 rows)
Python UDF: 1.46s  (1999500 rows)  → 14.5× slower


## 10 — `withColumn` in a loop is exponentially slow

Each `withColumn` call appends a node to the logical plan. The analyzer then re-traverses the **entire** plan to check column resolution and types. Calling `withColumn` 200 times in a Python loop produces a plan that's quadratic to analyze — the job can spend longer planning than running.

When adding many columns, build a single `select` with a list of expressions instead.


In [11]:
df = spark.range(1000)
N = 60   # 200+ is where it really stings; we keep it small for the demo

# Anti-pattern — withColumn in a loop
t0 = time.time()
chained = df
for i in range(N):
    chained = chained.withColumn(f"c{i}", F.col("id") + F.lit(i))
chained.count()
t_loop = time.time() - t0

# Fix — single select with all expressions
t0 = time.time()
exprs = [F.col("id")] + [(F.col("id") + F.lit(i)).alias(f"c{i}") for i in range(N)]
df.select(*exprs).count()
t_select = time.time() - t0

print(f"{N} withColumn calls : {t_loop:.2f}s")
print(f"Single select       : {t_select:.2f}s")
print(f"At N=200 the gap is dramatically wider — try it on a larger N.")


60 withColumn calls : 0.41s
Single select       : 0.08s
At N=200 the gap is dramatically wider — try it on a larger N.


## 11 — `repartition`, `coalesce`, `partitionBy` are three different things

People reach for whichever name they remember. They do very different jobs.

| API | What it changes | Triggers shuffle? | When to use |
|---|---|---|---|
| `df.repartition(n)` | In-memory partition count, even distribution | **Yes** (full shuffle) | Before a wide op; to balance skew |
| `df.repartition(n, "col")` | In-memory partitions hashed by column | **Yes** | Pre-shuffle for joins/aggregations on `col` |
| `df.coalesce(n)` | Reduces partition count by merging existing | **No** (narrow only) | After heavy filter; to write fewer files |
| `df.write.partitionBy("col")` | **On-disk** directory layout (`col=val/...`) | Yes during write | When the column has low cardinality and is frequently filtered |

Confusing `repartition` (in-memory) with `partitionBy` (on-disk) is one of the most common Spark mistakes. `partitionBy` is a property of the **write** path, not the DataFrame.


In [12]:
df = spark.range(1000).withColumn("bucket", F.col("id") % 4)

print(f"Default partitions      : {df.rdd.getNumPartitions()}")
print(f"After repartition(8)    : {df.repartition(8).rdd.getNumPartitions()}     (full shuffle)")
print(f"After coalesce(2)       : {df.coalesce(2).rdd.getNumPartitions()}      (no shuffle, may skew)")
print(f"After repartition('bucket'): {df.repartition('bucket').rdd.getNumPartitions()}  "
      f"(hash partitioned, one partition per bucket value)")

# write.partitionBy is on-disk only — unrelated to RDD partitions
import shutil, tempfile
out = tempfile.mkdtemp()
df.write.mode("overwrite").partitionBy("bucket").parquet(out)
print(f"\nOn-disk layout from write.partitionBy('bucket'):")
for entry in sorted(os.listdir(out))[:6]:
    print(f"  {entry}")
shutil.rmtree(out, ignore_errors=True)


Default partitions      : 12
After repartition(8)    : 8     (full shuffle)
After coalesce(2)       : 2      (no shuffle, may skew)
After repartition('bucket'): 1  (hash partitioned, one partition per bucket value)



On-disk layout from write.partitionBy('bucket'):
  ._SUCCESS.crc
  _SUCCESS
  bucket=0
  bucket=1
  bucket=2
  bucket=3


## 12 — `SparkSession.builder.getOrCreate()` ignores new configs if a session already exists

`getOrCreate()` does exactly what it says: if a session already exists in the JVM, **it returns that session**. Any `.config(...)` calls in your new builder are silently ignored. This bites notebook users who tweak a config in a later cell and wonder why nothing changed.

To apply a new static config (one that must be set at session creation), you must `spark.stop()` first.

Runtime configs — anything you can change via `spark.conf.set` after creation — can be updated on a live session without restart.


In [13]:
# Pretend an earlier cell created the session with shuffle.partitions=4 (we did, in setup)
existing = SparkSession.builder.getOrCreate()
print(f"shuffle.partitions seen by app: {existing.conf.get('spark.sql.shuffle.partitions')}")

# Try to set a different value via a new builder — silently ignored
try_new = (SparkSession.builder
    .config("spark.sql.shuffle.partitions", "999")
    .getOrCreate())
print(f"After builder.config('spark.sql.shuffle.partitions', '999').getOrCreate(): "
      f"{try_new.conf.get('spark.sql.shuffle.partitions')}  ← unchanged")

# Runtime configs CAN be changed on a live session
existing.conf.set("spark.sql.shuffle.partitions", "8")
print(f"After spark.conf.set(...) on the live session: "
      f"{existing.conf.get('spark.sql.shuffle.partitions')}  ← changed")
# Restore for the rest of the notebook
existing.conf.set("spark.sql.shuffle.partitions", "4")


shuffle.partitions seen by app: 4
After builder.config('spark.sql.shuffle.partitions', '999').getOrCreate(): 999  ← unchanged
After spark.conf.set(...) on the live session: 8  ← changed


## 13 — Classic builder ≠ Spark Connect builder

There are now **two** SparkSession.Builder classes in PySpark:

- `pyspark.sql.session.SparkSession.Builder` — **classic**, drives a local JVM
- `pyspark.sql.connect.session.SparkSession.Builder` — **Connect**, talks to a remote Spark server over gRPC

They look identical at the call site but are different types. Helpers like `delta.configure_spark_with_delta_pip(builder)` accept only the classic builder — handing them a Connect builder fails with:

```
This function must be called with a SparkSession builder as the argument.
The argument found is of type <class 'pyspark.sql.connect.session.SparkSession.Builder'>.
```

Why? Connect helpers can't inject JARs into a remote JVM — that has to be configured server-side at startup.

Things that flip you into Connect mode without you asking:
- Setting `SPARK_REMOTE` or `SPARK_CONNECT_MODE_ENABLED` in the environment
- Importing from `pyspark.sql.connect` explicitly
- Databricks Connect / databricks-connect installed and active
- A shim in your environment that reassigns `pyspark.sql = pyspark.sql.connect`


In [14]:
# Inspect the builder you actually have
print(f"Active session type : {type(spark).__module__}.{type(spark).__name__}")

b = SparkSession.builder
print(f"Builder type        : {type(b).__module__}.{type(b).__name__}")

# Detect Connect mode quickly
is_connect = "connect" in type(b).__module__
print(f"In Spark Connect mode? {is_connect}")

# If you must run Delta locally, force the classic path:
#   1) unset SPARK_REMOTE / SPARK_CONNECT_MODE_ENABLED
#   2) import explicitly: from pyspark.sql.session import SparkSession
#   3) pass that builder to configure_spark_with_delta_pip
print("\nEnvironment hints:")
for var in ("SPARK_REMOTE", "SPARK_CONNECT_MODE_ENABLED"):
    print(f"  {var} = {os.environ.get(var, '(not set)')}")


Active session type : pyspark.sql.session.SparkSession
Builder type        : pyspark.sql.session.Builder
In Spark Connect mode? False

Environment hints:
  SPARK_REMOTE = (not set)
  SPARK_CONNECT_MODE_ENABLED = (not set)


## 14 — Decimal arithmetic widens precision and can overflow to null

Spark's `DecimalType(p, s)` follows SQL precision-widening rules:

```
Decimal(10,2) + Decimal(10,2) → Decimal(11,2)
Decimal(10,2) * Decimal(10,2) → Decimal(21,4)
Decimal(38,2) * Decimal(38,2) → would need Decimal(77,4), capped at 38 → loss or null
```

Once a result exceeds `DecimalType`'s 38-digit max, Spark either rounds (with precision loss) or returns null, depending on `spark.sql.decimalOperations.allowPrecisionLoss` (default `true` — silently rounds; set `false` to get null on overflow).

Either way, you get the wrong answer with no exception.


In [15]:
schema = "amount DECIMAL(38, 18)"
big = spark.createDataFrame([(Decimal("9" * 20 + ".5"),)], schema)
print("Input:")
big.show(truncate=False)

# Multiply two huge decimals — result needs > 38 precision
overflowed = big.withColumn("squared", F.col("amount") * F.col("amount"))
print("After multiplication (allowPrecisionLoss=true, default):")
overflowed.show(truncate=False)
print(f"  result schema: {overflowed.schema['squared'].dataType}")

# Disable precision loss — silent rounding becomes silent null
spark.conf.set("spark.sql.decimalOperations.allowPrecisionLoss", "false")
strict = big.withColumn("squared", F.col("amount") * F.col("amount"))
print("With allowPrecisionLoss=false — overflow returns null:")
strict.show(truncate=False)
spark.conf.set("spark.sql.decimalOperations.allowPrecisionLoss", "true")


Input:
+---------------------------------------+
|amount                                 |
+---------------------------------------+
|99999999999999999999.500000000000000000|
+---------------------------------------+

After multiplication (allowPrecisionLoss=true, default):
+---------------------------------------+-------+
|amount                                 |squared|
+---------------------------------------+-------+
|99999999999999999999.500000000000000000|NULL   |
+---------------------------------------+-------+

  result schema: DecimalType(38,6)
With allowPrecisionLoss=false — overflow returns null:
+---------------------------------------+-------+
|amount                                 |squared|
+---------------------------------------+-------+
|99999999999999999999.500000000000000000|NULL   |
+---------------------------------------+-------+



## 15 — Date parsing got strict in Spark 3.0

Before Spark 3.0, the date parser was lenient and tolerated formats like `2024-1-1` or `2024/01/01` when you asked for `yyyy-MM-dd`. Since 3.0, the default parser is **strict** — it raises (or returns null, depending on the read mode) on anything that doesn't exactly match.

Three behaviours, controlled by `spark.sql.legacy.timeParserPolicy`:

| Value | Behaviour |
|---|---|
| `EXCEPTION` (default) | Strict — invalid inputs throw |
| `CORRECTED` | Strict — invalid inputs return null |
| `LEGACY` | Pre-3.0 lenient parser |

Hits people moving Spark 2.x code to 3.x — same data, suddenly null dates.


In [16]:
raw = spark.createDataFrame([("2024-01-15",), ("2024-1-7",), ("not-a-date",)], "s STRING")

# CORRECTED returns null on bad input (no exception)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")
print("CORRECTED policy (strict, nulls):")
raw.withColumn("parsed", F.to_date("s", "yyyy-MM-dd")).show()

# LEGACY accepts loose formats
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
print("LEGACY policy (lenient):")
raw.withColumn("parsed", F.to_date("s", "yyyy-MM-dd")).show()

spark.conf.set("spark.sql.legacy.timeParserPolicy", "EXCEPTION")
print("EXCEPTION policy (default) — would throw on 'not-a-date' or '2024-1-7'.")


CORRECTED policy (strict, nulls):


+----------+----------+
|         s|    parsed|
+----------+----------+
|2024-01-15|2024-01-15|
|  2024-1-7|      NULL|
|not-a-date|      NULL|
+----------+----------+

LEGACY policy (lenient):


+----------+----------+
|         s|    parsed|
+----------+----------+
|2024-01-15|2024-01-15|
|  2024-1-7|2024-01-07|
|not-a-date|      NULL|
+----------+----------+

EXCEPTION policy (default) — would throw on 'not-a-date' or '2024-1-7'.


## 16 — Joining on `df1.k == df2.k` keeps both `k` columns

Two different join syntaxes look interchangeable but produce different schemas:

```python
df1.join(df2, df1.k == df2.k)     # keeps BOTH k columns → ambiguous references later
df1.join(df2, "k")                 # keeps ONE k column (string or list of strings)
df1.join(df2, ["k1", "k2"])        # same, for compound keys
```

After the first form, `joined.select("k")` raises `AMBIGUOUS_REFERENCE`. You then have to qualify with `df1.k` — but if you `select(*)` and pass it downstream, you have two columns with identical names, and a later `groupBy("k")` is ambiguous too.


In [17]:
orders = spark.createDataFrame([("C1", "ORD1"), ("C2", "ORD2")], "customer_id STRING, order_id STRING")

# Anti-pattern — both customer_id columns survive
both = customers.join(orders, customers.customer_id == orders.customer_id)
print(f"Columns after expression-form join: {both.columns}")
try:
    both.select("customer_id").show()
except Exception as e:
    print(f"  select('customer_id') → AMBIGUOUS: {str(e)[:100]}")

# Cleaner — string form drops the duplicate
clean = customers.join(orders, on="customer_id")
print(f"\nColumns after string-form join: {clean.columns}")
clean.select("customer_id", "name", "order_id").show()


Columns after expression-form join: ['customer_id', 'name', 'state', 'credit_score', 'customer_id', 'order_id']
  select('customer_id') → AMBIGUOUS: [AMBIGUOUS_REFERENCE] Reference `customer_id` is ambiguous, could be: [`customer_id`, `customer_id`]

Columns after string-form join: ['customer_id', 'name', 'state', 'credit_score', 'order_id']


+-----------+-----+--------+
|customer_id| name|order_id|
+-----------+-----+--------+
|         C1|Alice|    ORD1|
|         C2|  Bob|    ORD2|
+-----------+-----+--------+



In [18]:
spark.stop()
